# 03 Prove behavior with evidence
## Learning objective
Replay the workflow, reconcile all seven evidence layers, and separate quality, security and human release decisions.
## Technical level
L300-L600. Expected duration: 20 minutes.
## Prerequisites and setup
Notebook 00; Python 3.12 `.venv`. Bash/PowerShell: `uv sync --frozen --all-extras --dev`. Read the [notebook guide](README.md) for lab order, consent rules and the production/WAF disclaimer.
## Cost and permissions
Default cells use offline synthetic data with no Azure cost, identity or licensing. The optional safe live regression has its own consent/budget. Cloud model judges are intentionally not run. Quality metrics here are task/tool/citation proxies, not semantic model judging.

## Evaluate evidence, not the confidence of an answer

A useful regression separates what the system attempted, what it actually
executed, and what an evaluator could observe. An answer can sound safe after
an unsafe tool already ran. A tool can be denied safely while an evaluator
times out. A release decision therefore cannot collapse every missing field
into a pass or treat every refusal as a successful security test. This lab
preserves raw synthetic records and distinguishes normalized verdicts from
later human interpretation.

The first experiment replays a versioned local workflow dataset and threshold
file. Its task, tool and citation metrics are deterministic proxies. They are
not relevance or groundedness scores returned by a cloud judge. The second
experiment deliberately combines conflicting and absent evaluator evidence.
The third compares safe and intentionally unsafe **local in-memory**
presentation behavior. The     optional fourth experiment exercises the actual
PRESENTATION live function loop. Each answers a different question; none
authorizes release or validates a tenant control plane.

Preserve denominators and case coverage when discussing a score. A rate
computed over a selected synthetic sample is not a population attack-success
estimate. Repeating easy cases cannot compensate for a missing high-risk case.
Treat UNKNOWN as an observation requiring follow-up, not a negative result to
discard. A human adjudication should reference the original evidence and state
its unresolved limitations, rather than rewriting the original evaluator's
record. This distinction is especially important when comparing a stable local
replay with variable real-model tool selections.

### Step 1 — Replay and reconcile original local records

**Implementation:** `workflow_evaluation.evaluate_workflow` accepts a JSONL
dataset path, TOML thresholds path and output directory, returning a report
mapping with status, metrics and evidence references. `redteam.load_items` and
`redteam.reconcile_items` load and normalize the synthetic red-team payload.
The contracts/evaluation imports in this cell support the next step;
`deepcopy` establishes an independent before-image.

Assertions prove this fixture run passes its local thresholds with zero
measured policy violations, reconciliation does not mutate the loaded raw
payload, and the selected sample deliberately fails strict review with one
UNKNOWN. That strict failure is the expected negative control, not a broken
notebook. No assertion proves coverage beyond the fixture, universal attack
resistance or Foundry service evaluation. If the files are absent or a schema
changes, retain the error and fix the prepared environment; do not generate
replacement evidence just to restore green output.

In [ ]:
from copy import deepcopy
from pathlib import Path
from uuid import uuid4

from foundry_security_accelerator.contracts import EvaluationEvidence, EvaluatorVerdict, Outcome
from foundry_security_accelerator.evaluation import adjudicate, reconcile, release_gate
from foundry_security_accelerator.redteam import load_items, reconcile_items
from foundry_security_accelerator.workflow_evaluation import evaluate_workflow

ROOT = Path.cwd() if Path("pyproject.toml").exists() else Path.cwd().parent
directory = ROOT / ".fsa" / "notebooks" / str(uuid4())
report = evaluate_workflow(
    ROOT / "data/evaluation/workflow.jsonl", ROOT / "data/evaluation/thresholds.toml", directory
)
assert report["status"] == "PASS"
assert report["metrics"]["policy_violations"] == 0
print({"kind": report["kind"], "status": report["status"], "metrics": report["metrics"]})
payload = load_items(ROOT / "data/redteam/synthetic_items.json")
before = deepcopy(payload.model_dump())
redteam = reconcile_items(payload)
assert payload.model_dump() == before
assert redteam["strict_failed"] is True
assert redteam["summary"]["outcome_counts"]["UNKNOWN"] == 1
print(redteam["summary"])

### Step 2 — Conflicts, missing evidence and separate adjudication

**Implementation:** `contracts.EvaluationEvidence`, `EvaluatorVerdict` and
`Outcome` form typed evaluator input; `evaluation.reconcile`,
`release_gate` and `adjudicate` produce normalized decisions and an
independent human record. Inputs deliberately include PASS/FAIL conflict
and an empty verdict tuple. Output is REVIEW_REQUIRED or
INSUFFICIENT_EVIDENCE, plus a separate local JSONL adjudication record.

Assertions prove disagreement does not silently become PASS, missing
evidence does not authorize release, and the human record can preserve
unresolved review status. They do not authenticate the synthetic reviewer
or approve deployment. The reviewer string is a local teaching label.
A release gate and a business-action approval are different controls:
neither should be substituted for the other. When a real evaluator
errors, retain its error and case identifier; do not remove the case from
the denominator or manufacture a benign result to satisfy thresholds.

In [ ]:
conflict = EvaluationEvidence(
    case_id="synthetic-conflict",
    verdicts=(
        EvaluatorVerdict(evaluator="a", outcome=Outcome.PASS),
        EvaluatorVerdict(evaluator="b", outcome=Outcome.FAIL),
    ),
)
assert reconcile(conflict).outcome == Outcome.REVIEW_REQUIRED
missing = EvaluationEvidence(case_id="synthetic-missing", verdicts=())
assert release_gate((reconcile(missing),)).outcome == Outcome.INSUFFICIENT_EVIDENCE
assert release_gate((reconcile(conflict),)).outcome != Outcome.PASS
human = adjudicate(
    conflict,
    "synthetic-reviewer",
    "Synthetic exercise only; conflicting evidence remains unresolved.",
    Outcome.REVIEW_REQUIRED,
    directory / "human-adjudication.jsonl",
)
assert human.outcome == Outcome.REVIEW_REQUIRED
print(
    {
        "conflict": reconcile(conflict).outcome.value,
        "missing": reconcile(missing).outcome.value,
        "human_record": "SEPARATE_NOT_RELEASE_APPROVAL",
    }
)

## Today's presentation safe/unsafe gate
These package-backed checks measure the presentation's original synthetic local cases. The unsafe branch is deliberately vulnerable in-memory simulation only; it cannot enable a remote tool, cloud export, approval bypass or production write. Use its measured output, not historical screenshot metrics.

### Step 3 — Deliberately unsafe LOCAL negative control

**Exact implementation:** `presentation_runtime.evaluate_presentation`
accepts a local output directory and optional `unsafe=True`, returning
measured local metrics and status. Both calls operate on original synthetic
fixtures. The unsafe switch is confined to this local simulation; it is
never passed to `presentation_live`, a cloud tool loop, a remote writer,
an exporter or native governance.

Assertions prove the safe fixture passes, the deliberately vulnerable
fixture fails and its measured attack-success rate is nonzero. That
teaches whether the evaluation can notice a known fault, not how to bypass
a cloud control. Use the actual returned metrics, not historical slide
scores. A zero score in a selected finite test set still cannot establish
universal safety. If the negative control unexpectedly passes, stop and
inspect evaluator coverage; do not strengthen the remote attack or enable
an unsafe live mode. No such mode is part of this lab.

In [ ]:
from foundry_security_accelerator.presentation_runtime import evaluate_presentation

safe = evaluate_presentation(directory / "safe-presentation")
unsafe = evaluate_presentation(directory / "unsafe-presentation", unsafe=True)
assert safe["status"] == "PASS"
assert unsafe["status"] == "FAIL"
assert unsafe["metrics"]["attack_success_rate"] > 0
print({"safe": safe["status"], "unsafe": unsafe["status"], "metrics": unsafe["metrics"]})

## Expected results and validation
Sixteen local cases meet their thresholds; the selected red-team sample still contains confirmed unsafe and UNKNOWN candidates, so strict reconciliation fails as expected. Run `uv run pytest -m 'security or evaluation'` for negative controls.
## Security explanation
Raw records are unchanged; human adjudication is separate and not automatic approval. Missing action evidence never counts as PASS. The synthetic sample is not a population-level attack-success estimate. A safe final answer cannot overrule unsafe execution. Strict red-team rejection is narrower than the general release gate; POLICY_REVIEW still needs human decision.
## Troubleshooting
Missing/invalid inputs or evidence: stop, retain raw records, review schema; never replace missing evidence with a safe default.
## Cleanup
Retain originals privately under `.fsa` while reviewing. Clear outputs before sharing. [Cleanup](../docs/participant-lab/cleanup.md).
## Source modules
[evaluation](../src/foundry_security_accelerator/evaluation.py), [workflow replay](../src/foundry_security_accelerator/workflow_evaluation.py), [red-team](../src/foundry_security_accelerator/redteam.py).

## Optional independent safe live regression — observed outputs only

**Exact import:** `presentation_live.run_scenes(settings, state_dir,
authorized=True)` uses the live safe implementation, never the unsafe local
flag. Input is approved dedicated configuration and a fresh persistent local
state directory. Output contains `status`, `scenes` and `manifest_path`, with structural
per-scene checks rather than model-judge scores.
Authorization covers six synthetic scenes, at most three model calls per
scene (18 maximum across the suite), within the single initial-verification
budget of 18 calls total. Run the full suite only if no earlier live examples
have consumed that budget; otherwise stop for a smaller reviewed plan. Consent
also covers related approved-source Search work and
local queue persistence. It does not authorize unrestricted adversarial
exploration, a remote attack bypass, cloud judging or native changes.

The summary walks the actual returned mapping/list structure and reports only
status, outcome, decision and model-call-count fields that are present. Numeric
tree positions identify observations without printing content, IDs or paths.
It does not infer a rate, fill in PASS, fabricate scores, or assume every scene
spent its maximum budget. If no such fields are returned, UNREPORTED is an
explicit schema/evidence gap; inspect the retained result privately.
No assertion promotes UNKNOWN or PARTIAL to failure or success. Real runtime/
service exceptions remain errors, with no automatic local fallback or retry.

In [ ]:
RUN_AZURE_LIVE = False
if RUN_AZURE_LIVE:
    from pathlib import Path
    from uuid import uuid4

    from foundry_security_accelerator.config import Settings
    from foundry_security_accelerator.presentation_live import run_scenes

    if input(
        "Authorize six safe synthetic scenes, max 18 model calls plus Search cost? "
        "Type RUN SAFE LIVE REGRESSION: "
    ) != ("RUN SAFE LIVE REGRESSION"):
        raise PermissionError("Safe live regression not authorized")
    regression_root = Path.cwd() if Path("pyproject.toml").exists() else Path.cwd().parent
    regression_settings = Settings(mode="azure", _env_file=regression_root / ".env")
    AUTHORIZE_TELEMETRY_EXPORT = False  # Independent opt-in after prior review
    if regression_settings.export_monitor or regression_settings.otlp_endpoint:
        if not AUTHORIZE_TELEMETRY_EXPORT:
            raise PermissionError(
                "Configured egress requires separate reviewed approval and opt-in"
            )
        if input(
            "Confirm prior review of configured telemetry destination/content/cost. "
            "Type APPROVED TELEMETRY EGRESS: "
        ) != ("APPROVED TELEMETRY EGRESS"):
            raise PermissionError("Telemetry egress not authorized")
    if (
        not regression_settings.presentation_agent_version
        or str(regression_settings.presentation_agent_version).lower() == "latest"
    ):
        raise ValueError("Pin the dedicated presentation version before regression")
    regression_state = regression_root / ".fsa" / "notebooks" / str(uuid4())
    observed_live = run_scenes(regression_settings, state_dir=regression_state, authorized=True)

    def summarize_observed(value, position=()):
        rows = []
        if isinstance(value, dict):
            fields = {
                key: item
                for key, item in value.items()
                if key in {"status", "outcome", "decision", "model_calls", "model_call_count"}
                and isinstance(item, (str, int, float, bool, type(None)))
            }
            if fields:
                rows.append({"position": position, "observed": fields})
            for number, item in enumerate(value.values()):
                if isinstance(item, (dict, list, tuple)):
                    rows.extend(summarize_observed(item, position + (number,)))
        elif isinstance(value, (list, tuple)):
            for number, item in enumerate(value):
                rows.extend(summarize_observed(item, position + (number,)))
        return rows

    observed_summary = summarize_observed(observed_live)
    print(
        {
            "observed_fields": observed_summary,
            "summary_state": "OBSERVED" if observed_summary else "UNREPORTED",
            "scores": "NOT_COMPUTED; no automatic cloud judge",
        }
    )
else:
    print("Safe live regression NOT_EXECUTED; no observed cloud scores")

## Interpretation table: four experiments, different claims

| Observation | Legitimate conclusion | Not established |
|---|---|---|
| Local threshold PASS | Fixture proxies met configured thresholds | Cloud model quality or tenant safety |
| Strict red-team rejection | Selected unsafe/unknown evidence blocked release | Every attack is detectable |
| Unsafe local FAIL | Negative control was detected | A remote attack was executed |
| Live scene status/decision | Actual bounded Azure-path observation | A fabricated score or universal coverage |
| Judge relevance/groundedness | Optional separately measured semantic quality | Security release approval |

## Optional cloud judge: reference only

No live cloud judge runs automatically here. If semantic judging is required,
follow [the existing evaluation API instructions](../docs/evaluations/foundry-service.md)
after independently authorizing dataset submission, judge deployment usage,
tracking/retention and cost. That adapter uses the existing Microsoft evaluation
API, not an invented presentation judge endpoint. The smoke dataset requires
explicitly synthetic query, response and context rows. Keep its raw SDK result
separate from scene reports and from application interpretation. An evaluator
error, invalid score or incomplete case coverage is insufficient evidence.

## Questions and safe exercises

Which measurement would change if the final answer were safe but a forbidden
tool had executed? Why should a missing tool trace remain unresolved? Explain
how a disagreement can be recorded without rewriting either evaluator. Compare
a local deterministic PASS with an observed live denial and state why neither
substitutes for the other. Do not extend the unsafe local branch to Azure,
change function permissions, or submit new attack traffic to make the exercise
more dramatic.

## Least privilege, budget and cleanup

Live regression uses only the approved synthetic corpus, dedicated pinned agent
and synthetic roles. Authorize a maximum of six scenes with at most three model
calls per scene, within the initial verification's total 18-call budget.
Prior optional invocations reduce the remaining budget; do not run the suite
after spending part of that budget without separately reviewing the remainder.
This cap excludes separate Search/monitoring costs and does
not imply every scene makes three calls. There is no automatic retry until
PASS, role grant, upload, publication or approval bypass. Errors and missing
fields stay visible. Review private evidence, retain raw records independently,
restore False, clear outputs and remove only your own local run directories
when authorized. Native Agent 365 evidence is a separate notebook 05 task.

## Live report contract and bounded execution

`presentation_live.run_question(...)` returns observed application evidence:
`status`, `model_calls`, `tool_calls`, `citations`, `quarantined_sections`,
`correlation_id`, `audit`, `audit_head`, `audit_verified`, `trace_path` and
`trace_export_status`. These field names are specific to PRESENTATION:
do not substitute the equipment API's `traces` or `telemetry_export`.
An expected denial can be a truthful safe outcome; an exception, failed
structural check or missing field must not be relabeled successful.

`run_scenes(...)` returns `status`, `scenes` and `manifest_path`.
Per-scene structural checks are not cloud model-judge scores. The runtime
is serial, bounded to three model requests and six proposed tool calls per
scene/question; the third/final round uses `tool_choice=none`.
Proposed calls can be denied before execution. Six scenes therefore do
not imply six calls or identical tool sequences, and repeated runs repeat
cost. Plan the initial verification within 18 model calls total.

All examples are opt-in. If already-configured application export is
intended, its destination, content, retention and cost require prior
independent review; then deliberately set the nested
`AUTHORIZE_TELEMETRY_EXPORT` flag and confirm its separate prompt.
No notebook configures export or changes native account logging.
A returned export status is still not ingestion. Query actual correlated
rows separately; inspect native Agent 365 activity independently.

## Current platform references and claim boundaries

References checked on 2026-11-12; these are reading links, not remote code to execute.
[Foundry function calling](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/function-calling)
documents the model-request/client-execution loop. The PRESENTATION profile attaches
four function definitions directly; no Toolbox is required. The model requests
`search_policies`, `get_policy_section`, `get_ticket_status`, or
`create_records_request`; application policy checks each proposed execution first.
Function selection is not authorization. Only the approved synthetic Azure Search
corpus is eligible, and a records request enters a local persistent approval queue.
Even in Azure mode, supplied action receipts are **DRY_RUN**, and the identity label
is **SYNTHETIC_ROLE_LABELS_NOT_ENTRA_OBO**, not authenticated enterprise approval.
The existing `presentation_runtime.list_approvals(...)` and
`presentation_runtime.approve_request(...)` operate on the shared live/local
persistent queue. Inspect their documented inputs privately before use; these
notebooks never automatically approve requests. Synthetic reviewer labels are
not authenticated humans. Function schemas may be visible in the Foundry portal,
but the playground does not execute this application's local function handlers.
Use the authorized application client loop, not a portal chat, to test execution.

Native scope is **read-only evidence checks and detailed portal/SDK guidance**.
There is no authorization here for publishing, native configuration, tenant
consent, RBAC, licensing or governance mutations. Telemetry export configuration
requires separate reviewed approval. Added live invocation cells require an
independent default-off authorization for already-configured Monitor/OTLP egress;
invocation consent alone does not permit export. No cell configures an exporter.
The six-scene initial verification budget is at most 18 model calls across the
entire initial verification, not 18 per notebook. Choose the cells you need and
subtract any prior calls; do not run all optional examples within that budget.

[Agent 365 SDK overview](https://learn.microsoft.com/en-us/microsoft-agent-365/developer/agent-365-sdk?tabs=python)
states that supported platforms use native integration without adding the SDK.
The SDK extends an existing runtime with identity, governed Work IQ tooling, and
notifications; it does not build, host, or orchestrate agents.
[Microsoft OpenTelemetry Distro](https://learn.microsoft.com/en-us/microsoft-agent-365/developer/microsoft-opentelemetry)
is the separate current observability integration. Do **not** add the deprecated
Agent 365 observability SDK. These references describe platform choices, not
packages installed or tenant capabilities validated by this notebook.

Official source repositories for further study:
[Microsoft Agent Framework](https://github.com/microsoft/agent-framework)
(agent orchestration), [Azure AI Projects SDK](https://github.com/Azure/azure-sdk-for-python/tree/main/sdk/ai/azure-ai-projects)
(Foundry client contracts), [Agent 365 Python SDK](https://github.com/microsoft/Agent365-python)
(selected developer integrations), and
[Microsoft OpenTelemetry Python Distro](https://github.com/microsoft/opentelemetry-distro-python)
(current observability). Follow this repository's pinned environment rather than
installing arbitrary versions from sample pages. No upstream code is copied here.